In [ ]:
# Step 0 - setup. Set CODE below, then run cells in order.
# One-time install if needed:
# %pip install -U google-genai pandas openpyxl
import os
import hashlib, json, re, time
from pathlib import Path
from typing import Any

import pandas as pd
from google import genai
from google.genai import types

# ----------------------------------------------------------------- config
API_KEY = "YOUR_GEMINI_API_KEY"          # <<< your Gemini API key
MODEL = "gemini-3.6-flash"              # coder model
TEMPERATURE = 0.1
MAX_OUTPUT_TOKENS = 4096                # cap only; leaves room for dynamic thinking + the digit

CODE = "B"                              # <<< set to "B", "C", or "D", then run cells 1-3

# Input: the trial-level data file of this repository. Rows whose trace fails
# the 0.15 visibility filter (trace_included == 0) are skipped.
ROOT_PATH = ".."
INPUT_FILE  = f"{ROOT_PATH}/results/trials.xlsx"
OUTPUT_FILE = f"{ROOT_PATH}/results/recoded_trials.xlsx"   # B / C / D codes accumulate into this file
TRACE_FIELD = "q1_trace"

BATCH_DIR = Path("batch"); BATCH_DIR.mkdir(exist_ok=True)
BATCH_TAG = "run3"                      # source tag for cells 1-3
RETRY_TAG = "retry1"                    # written by the retry cell
DOWNLOAD_RESULTS = True

# ----------------------------------------------------------------- prompts (placeholders filled per row)
PROMPT_B = """You are a qualitative coder. Read the reasoning trace below and answer one question.

The sentence under discussion is:
"{sentence}"

Question: What is the trace's final verdict on the structure of this sentence?

Judge by where the trace ENDS UP, not by moments of confusion along the way.

- 0: No difficulty. The trace parses the sentence smoothly and never flags any
  structural problem, confusion, or ungrammaticality.
- 1: Difficulty resolved as valid. The trace notices structural difficulty,
  ambiguity, confusion, or a missing/omitted comma, but ultimately concludes the
  sentence IS a complete, valid sentence with a correct parse (e.g. "this looks
  broken at first, but it is actually a well-formed garden-path sentence").
  Noting only a missing or omitted comma, or calling the sentence merely
  "awkward" or "confusing", while still parsing it, counts as 1.
- 2: Treated as ungrammatical. The trace treats the sentence as ungrammatical, a
  fragment, broken, defective, incomplete, a run-on, an error, or as needing to
  be fixed/rewritten, AND does not retract this by concluding it is actually
  valid. Code 2 even if:
    - the claim is hedged ("possibly a fragment", "seems incomplete"), and
    - the trace still extracts an interpretation or answers the question
      ("despite being a fragment, it clearly says X").
  Extracting meaning from a sentence the trace regards as defective does NOT
  upgrade it to 1.
- 3: Cannot decide.

Notes:
- Calling the sentence "confusing", "awkward", or "a garden-path" is not by
  itself a verdict of ungrammaticality. Code 2 only if the trace treats the
  sentence itself as invalid.
- If the trace raises a defect (fragment, broken, incomplete, ungrammatical) and
  never retracts it, code 2, even if it still answers. Code 1 only when the trace
  explicitly resolves the sentence as valid/complete. A missing or omitted comma
  is not a defect for this purpose; parsing despite it is 1.

You MUST respond with ONLY a single digit: 0, 1, 2, or 3. No words, no explanation."""

PROMPT_C = """You are a qualitative coder. Read the reasoning trace below and answer one question.

The sentence under discussion is:
"{sentence}"

The target verb is "{verb}", the verb of the subordinate clause (the clause
introduced by "while" or "as").

Question: How does the trace treat the object of "{verb}"?

An "object expectation" is any sign that the trace treats "{verb}" as needing,
taking, or missing an object: it considers the following noun phrase as the
object, or it frames the absent object as a gap (e.g. "it doesn\'t say what he
was hunting", "the object is unstated", "hunted something unspecified").

- 0: No object expectation. The trace consistently accepts the objectless
  reading of "{verb}" (intransitive, or self-directed for verbs like "washed"
  = washed himself) and never treats the missing object as an issue.
- 1: Object expectation appears but is dropped. The trace considers the noun
  phrase as a possible object, or flags the unstated object, but ultimately
  lets this go and settles on the objectless reading.
- 2: Object expectation persists. In the trace\'s final analysis, "{verb}" still
  has an object in play: it maintains that something unstated was acted on
  (e.g. "he was hunting something, we just aren\'t told what"), keeps the
  object as implicit or missing, or attaches the noun phrase as the object.
- 3: Cannot decide.

Examples:
- "the verb is used intransitively here" / "the thief hid (himself)" and the
  object question never arises -> 0
- "at first it looks like \'the deer\' is the object, but it is actually the
  subject of the main clause; \'hunted\' is intransitive here" -> 1
- "it doesn\'t specify what he was hunting, so we can\'t say it was the deer" ->
  2 (the object is still treated as existing but unstated)
- "he was hunting, and the deer was probably what he hunted" -> 2

You MUST respond with ONLY a single digit: 0, 1, 2, or 3. No words, no explanation."""

PROMPT_D = """You are a qualitative coder. Read the reasoning trace below and answer one question.

The sentence under discussion is:
"{sentence}"

Question: Does the trace use pragmatic inference to fill in or override what the
syntax leaves open, in a way that supports its answer?

Code 1 ONLY if BOTH of the following hold:
(a) The trace acknowledges, explicitly or implicitly, that the syntax alone does
    not state the relevant relation (e.g. the sentence does not say what was
    hunted, or the two clauses are formally separate), AND
(b) The trace then uses context, plausibility, implication, or common sense to
    conclude that the relation holds anyway, and this inference supports the
    answer it gives (e.g. "he was hunting something, and the deer is the obvious
    candidate, so yes, he hunted the deer" or "they are separate clauses, but the
    natural implication is that the deer was what he hunted").

- 0: No. The trace answers from the syntactic parse alone. This includes traces
  that use pragmatic-sounding vocabulary ("the natural reading", "plausibly",
  "common sense") merely to describe or justify the syntactic parse itself,
  without inferring anything beyond what the syntax states.
- 1: Yes. The trace makes the bridging move described above: pragmatics supplies
  a relation the syntax does not state, and that inference feeds the answer.
- 2: Cannot decide.

Key distinction:
- "Syntactically, the deer is the subject of \'ran\', so the sentence does not say
  he hunted the deer. Answer: No." -> 0 (syntax only)
- "The most natural reading is that the deer ran into the woods." -> 0
  (pragmatic vocabulary, but describing the parse, no bridging inference)
- "The sentence doesn\'t state the object of \'hunted\', but in context the deer is
  clearly what he was hunting. Answer: Yes." -> 1

You MUST respond with ONLY a single digit: 0, 1, or 2. No words, no explanation."""

CODE_CONFIG = {
    "B": {"prompt": PROMPT_B, "slots": ("sentence",),        "enum": ["0", "1", "2", "3"], "column": "B"},
    "C": {"prompt": PROMPT_C, "slots": ("sentence", "verb"), "enum": ["0", "1", "2", "3"], "column": "C"},
    "D": {"prompt": PROMPT_D, "slots": ("sentence",),        "enum": ["0", "1", "2"],      "column": "D"},
}
if CODE not in CODE_CONFIG:
    raise ValueError("CODE must be 'B', 'C', or 'D'.")
CFG = CODE_CONFIG[CODE]

# ----------------------------------------------------------------- subordinate-verb lookup (GP-invariant)
VERB_LOOKUP = {
    (1, "OT"): "hunted",   (2, "OT"): "sailed",      (3, "OT"): "photographed",
    (4, "OT"): "performed", (5, "OT"): "read",        (6, "OT"): "ordered",
    (7, "OT"): "wrote",     (8, "OT"): "typed",       (9, "OT"): "steered",
    (10, "OT"): "studied",  (11, "OT"): "whittled",   (12, "OT"): "drove",
    (13, "RAT"): "bathed",  (14, "RAT"): "groomed",   (15, "RAT"): "dried off",
    (16, "RAT"): "woke up", (17, "RAT"): "hid",       (18, "RAT"): "dressed",
    (19, "RAT"): "washed",  (20, "RAT"): "settled down", (21, "RAT"): "undressed",
    (22, "RAT"): "shaved",  (23, "RAT"): "scratched", (24, "RAT"): "calmed down",
}

def subordinate_verb(itemno, verb_type) -> str:
    v = VERB_LOOKUP.get((int(itemno), str(verb_type)))
    if v is None:
        raise KeyError(f"No verb for itemno={itemno}, verb_type={verb_type}")
    return v

def build_system_prompt(row) -> str:
    slots = {}
    if "sentence" in CFG["slots"]:
        slots["sentence"] = str(row["sent"])
    if "verb" in CFG["slots"]:
        slots["verb"] = subordinate_verb(row["itemno"], row["verb_type"])
    return CFG["prompt"].format(**slots)

def make_custom_id(itemno, verb_type, gp, model, trial) -> str:
    raw = f"{itemno}-{verb_type}-{gp}-{model}-trial{trial}"
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", raw)

def build_generation_config() -> dict:
    # No thinking_config => default dynamic thinking (medium for gemini-3.6-flash), thoughts not returned.
    # Enum-constrained output guarantees a single valid code from CFG["enum"].
    return {
        "temperature": TEMPERATURE,
        "response_mime_type": "text/x.enum",
        "response_schema": {"type": "STRING", "enum": CFG["enum"]},
        "max_output_tokens": MAX_OUTPUT_TOKENS,
    }

def build_batch_line(custom_id: str, system_text: str, trace_text: str) -> dict:
    user_text = f"Here is the reasoning trace to code:\n\n{trace_text}"
    return {
        "key": custom_id,
        "request": {
            "contents": [{"role": "user", "parts": [{"text": user_text}]}],
            "system_instruction": {"parts": [{"text": system_text}]},
            "generation_config": build_generation_config(),
        },
    }

# ----------------------------------------------------------------- response helpers
def get_field(value: Any, *names: str, default=None):
    if value is None:
        return default
    for name in names:
        if isinstance(value, dict) and name in value:
            return value[name]
        if hasattr(value, name):
            return getattr(value, name)
    return default

def extract_code(response: Any):
    """Return the integer code from an enum response, or None if unparseable."""
    candidates = get_field(response, "candidates", default=[]) or []
    if not candidates:
        return None
    parts = get_field(get_field(candidates[0], "content"), "parts", default=[]) or []
    for part in parts:
        if get_field(part, "thought", default=False):
            continue
        text = (get_field(part, "text", default="") or "").strip().rstrip(".")
        if text == "":
            continue
        try:
            return int(text)
        except ValueError:
            digits = [c for c in text if c.isdigit()]
            return int(digits[0]) if len(digits) == 1 else None
    if hasattr(response, "text") and response.text:
        t = response.text.strip()
        return int(t) if t.isdigit() else None
    return None

def result_key(item: dict):
    return item.get("key") or item.get("metadata", {}).get("key")

def load_result_items(client, batch, raw_results_file: Path):
    dest = getattr(batch, "dest", None)
    result_file_name = get_field(dest, "file_name", "fileName")
    if not result_file_name:
        raise RuntimeError("This batch has no result file. Check job state / destination.")
    print(f"Downloading result file: {result_file_name}")
    content = client.files.download(file=result_file_name)
    text = content if isinstance(content, str) else bytes(content).decode("utf-8")
    raw_results_file.write_text(text, encoding="utf-8")
    return [json.loads(line) for line in text.splitlines() if line.strip()]

print(f"Ready: CODE={CODE} -> column '{CFG['column']}', enum={CFG['enum']}, model={MODEL}")


In [ ]:
# Step 1 - build JSONL requests (review the file before submitting in cell 2)
df = pd.read_excel(INPUT_FILE, sheet_name="trials").rename(
    columns={"sentence": "sent", "condition": "GP", "trace_included": "include"})
for col in ("itemno", "verb_type", "GP", "model", "trial", "sent", "include", TRACE_FIELD):
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}' in {INPUT_FILE}. Columns: {list(df.columns)}")

def _is_true(v):
    return v is True or str(v).strip().lower() in {"true", "1", "yes", "y"}

included = df[df["include"].map(_is_true)].copy()
print(f"{len(included)} of {len(df)} rows included (trace_ratio >= 0.15).")

batch_file = BATCH_DIR / f"requests_{CODE}_{BATCH_TAG}.jsonl"
index_file = BATCH_DIR / f"index_{CODE}_{BATCH_TAG}.json"

request_index = []
skipped_empty = 0
with batch_file.open("w", encoding="utf-8") as h:
    for _, row in included.iterrows():
        trace = row[TRACE_FIELD]
        if not isinstance(trace, str) or not trace.strip():
            skipped_empty += 1
            continue
        cid = make_custom_id(row["itemno"], row["verb_type"], row["GP"], row["model"], row["trial"])
        system_text = build_system_prompt(row)
        request_index.append({
            "custom_id": cid,
            "itemno": int(row["itemno"]), "verb_type": str(row["verb_type"]),
            "GP": str(row["GP"]), "model": str(row["model"]), "trial": int(row["trial"]),
        })
        h.write(json.dumps(build_batch_line(cid, system_text, trace), ensure_ascii=False) + "\n")

index_file.write_text(json.dumps(request_index, indent=2, ensure_ascii=False), encoding="utf-8")
payload_mb = batch_file.stat().st_size / (1024 * 1024)
print(f"Wrote {len(request_index)} requests ({payload_mb:.2f} MB) -> {batch_file}")
if skipped_empty:
    print(f"Skipped {skipped_empty} rows with empty {TRACE_FIELD}.")
print(f"Index: {index_file}")
print("Review the JSONL, then run cell 2 to submit.")


In [ ]:
# Step 2 - upload and submit the batch (this makes the API request)
batch_file = BATCH_DIR / f"requests_{CODE}_{BATCH_TAG}.jsonl"
if not batch_file.exists() or batch_file.stat().st_size == 0:
    raise FileNotFoundError(f"Run cell 1 first; missing/empty: {batch_file}")

client = genai.Client(api_key=API_KEY)
print(f"Uploading {batch_file} ({batch_file.stat().st_size/(1024*1024):.2f} MB)...")
uploaded = client.files.upload(
    file=str(batch_file),
    config=types.UploadFileConfig(display_name=f"gpcode_{CODE}_{BATCH_TAG}", mime_type="jsonl"),
)
print(f"Uploaded: {uploaded.name}")

batch = client.batches.create(
    model=MODEL, src=uploaded.name,
    config={"display_name": f"gpcode_{CODE}_{BATCH_TAG}"},
)
job_file = BATCH_DIR / f"job_{CODE}_{BATCH_TAG}.txt"
job_file.write_text(batch.name, encoding="utf-8")
state = getattr(getattr(batch, "state", None), "name", getattr(batch, "state", ""))
print(f"Submitted. Job: {batch.name}  Status: {state}")
print(f"Saved job id -> {job_file}")


In [ ]:
# Step 3 - poll status; when succeeded, download and merge codes into OUTPUT_FILE
job_file = BATCH_DIR / f"job_{CODE}_{BATCH_TAG}.txt"
if not job_file.exists():
    raise FileNotFoundError(f"No job id: {job_file}. Run cell 2 first.")

client = genai.Client(api_key=API_KEY)
batch = client.batches.get(name=job_file.read_text(encoding="utf-8").strip())
state = getattr(getattr(batch, "state", None), "name", getattr(batch, "state", ""))
print(f"Job: {batch.name}  Status: {state}")

if state != "JOB_STATE_SUCCEEDED":
    if state in {"JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}:
        print(f"Ended without success. Error: {getattr(batch, 'error', '')}")
    else:
        print("Not finished yet. Re-run this cell later.")
elif not DOWNLOAD_RESULTS:
    print("Succeeded. Set DOWNLOAD_RESULTS = True (cell 0) and re-run to export.")
else:
    index_file = BATCH_DIR / f"index_{CODE}_{BATCH_TAG}.json"
    request_index = json.loads(index_file.read_text(encoding="utf-8"))
    raw_results_file = BATCH_DIR / f"results_{CODE}_{BATCH_TAG}.jsonl"
    items = load_result_items(client, batch, raw_results_file)
    items_by_key = {k: it for it in items if (k := result_key(it))}
    print(f"Downloaded {len(items)} result lines for {len(request_index)} requests.")

    codes, n_ok, n_bad = {}, 0, 0
    for i, info in enumerate(request_index):
        item = items_by_key.get(info["custom_id"])
        if item is None and i < len(items):
            item = items[i]
        code = None
        if item is not None and not (item.get("error") or item.get("status")):
            code = extract_code(item.get("response"))
        codes[info["custom_id"]] = code
        n_ok += code is not None
        n_bad += code is None
    print(f"Parsed codes: {n_ok} ok, {n_bad} missing/failed.")

    # Merge into the accumulating output file (create from input on first code).
    base = OUTPUT_FILE if os.path.exists(OUTPUT_FILE) else INPUT_FILE
    out = pd.read_excel(base)
    col = CFG["column"]
    if col not in out.columns:
        out[col] = pd.NA

    def _row_cid(r):
        return make_custom_id(r["itemno"], r["verb_type"], r["GP"], r["model"], r["trial"])

    cid_series = out.apply(_row_cid, axis=1)
    out[col] = cid_series.map(codes).where(cid_series.isin(codes), out[col])
    out.to_excel(OUTPUT_FILE, index=False)
    filled = out[col].notna().sum()
    print(f"Wrote column '{col}' ({filled} coded rows) -> {OUTPUT_FILE}")


In [ ]:
# Step 4 (optional) - retry requests that returned no usable code
source_index = BATCH_DIR / f"index_{CODE}_{BATCH_TAG}.json"
source_job = BATCH_DIR / f"job_{CODE}_{BATCH_TAG}.txt"
if not source_index.exists() or not source_job.exists():
    raise FileNotFoundError("Missing source index/job. Run cells 1-3 first.")

client = genai.Client(api_key=API_KEY)
batch = client.batches.get(name=source_job.read_text(encoding="utf-8").strip())
state = getattr(getattr(batch, "state", None), "name", getattr(batch, "state", ""))
if state != "JOB_STATE_SUCCEEDED":
    raise RuntimeError("Source batch not succeeded yet.")

original_index = json.loads(source_index.read_text(encoding="utf-8"))
raw_results_file = BATCH_DIR / f"results_{CODE}_{BATCH_TAG}.jsonl"
items = load_result_items(client, batch, raw_results_file)
items_by_key = {k: it for it in items if (k := result_key(it))}

df = pd.read_excel(INPUT_FILE)

# Map custom_id -> row for rebuilding requests
rows_by_cid = {}
for _, row in df.iterrows():
    cid = make_custom_id(row["itemno"], row["verb_type"], row["GP"], row["model"], row["trial"])
    rows_by_cid[cid] = row

missing = []
for i, info in enumerate(original_index):
    item = items_by_key.get(info["custom_id"]) or (items[i] if i < len(items) else None)
    ok = item is not None and not (item.get("error") or item.get("status")) \
         and extract_code(item.get("response")) is not None
    if not ok:
        missing.append(info["custom_id"])
print(f"To retry: {len(missing)} of {len(original_index)}")

retry_file = BATCH_DIR / f"requests_{CODE}_{RETRY_TAG}.jsonl"
retry_index_file = BATCH_DIR / f"index_{CODE}_{RETRY_TAG}.json"
retry_index = []
with retry_file.open("w", encoding="utf-8") as h:
    for info in original_index:
        if info["custom_id"] not in missing:
            continue
        row = rows_by_cid[info["custom_id"]]
        system_text = build_system_prompt(row)
        h.write(json.dumps(build_batch_line(info["custom_id"], system_text, row[TRACE_FIELD]), ensure_ascii=False) + "\n")
        retry_index.append(info)
retry_index_file.write_text(json.dumps(retry_index, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {len(retry_index)} retry requests -> {retry_file}")
print(f"To submit: set BATCH_TAG = {RETRY_TAG!r} in cell 0, rerun cell 0, then run cell 2, then cell 3.")
